In [ ]:
# !pip install pandas plotly requests
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import requests


DATA_DIR = Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Más información:
- [6.1 Rate of invasive alien species establishment](https://drive.google.com/drive/u/0/folders/1jOzMGO5YSKRVAB6K8JyEa1l6i7CaA6vm)
- [datos](https://drive.google.com/drive/u/0/folders/1vbfu_PNFdEdclxO-DbocAz19SrpIS-kB)

## Descarga data desde GBIF

In [ ]:
def download_data_from_gbif(url: str, output_dir: Path) -> Path:
    file_name = url.split("?r=")[-1] + ".zip"
    response = requests.get(url, stream=True)
    with open(output_dir / file_name, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)

    return output_dir / file_name


def unzip_file(zip_file_path: Path, extract_folder: Path):
    with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)


urls = [
    "https://cloud.gbif.org/griis/archive.do?r=juan-fernandez-islands-chile-griis",
    "https://cloud.gbif.org/griis/archive.do?r=chile-griis-gbif",
    "https://cloud.gbif.org/griis/archive.do?r=rapa-nui-chile-griis-gbif",
]

for url in urls:
    file_name = download_data_from_gbif(url, DATA_DIR)
    unzip_file(zip_file_path=file_name, extract_folder=DATA_DIR / str(file_name.stem))

## Descarga data desde Zenodo

In [ ]:
def download_zenodo_dataset(url: str, output_dir: Path, file_name: str):
    response = requests.get(url, stream=True)
    with open(output_dir / file_name, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            f.write(chunk)


url = "https://zenodo.org/records/10039630/files/GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx?download=1"
download_zenodo_dataset(
    url=url,
    output_dir=DATA_DIR,
    file_name="GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx",
)

In [7]:
df_alien_species = pd.read_excel(
    DATA_DIR / "GlobalAlienSpeciesFirstRecordDatabase_v3.1_freedata.xlsx",
    sheet_name="FirstRecords",
    dtype={
        "TaxonName": "category",
        "OrigName": "category",
        "scientificName": "category",
        "FirstRecord_orig": str,
        "Family": "category",
        "Order": "category",
    },
)

df_alien_species_chile = (
    df_alien_species[df_alien_species["Region"] == "Chile"]
    .reset_index(drop=True)
    .copy()
)

In [8]:
# df_alien_species_chile[df_alien_species_chile["FirstRecord"] >= 1970].hist()
fig = px.histogram(
    df_alien_species_chile,
    x="FirstRecord",
    title=f"Primer año de registro de especies invasoras entre {df_alien_species_chile['FirstRecord'].min()} y {df_alien_species_chile['FirstRecord'].max()}",
)
fig.show()

In [9]:
fig = px.histogram(
    df_alien_species_chile[df_alien_species_chile["FirstRecord"] >= 1970],
    x="FirstRecord",
    title=f"Primer año de registro de especies invasoras entre {df_alien_species_chile[df_alien_species_chile['FirstRecord'] >= 1970]['FirstRecord'].min()} y {df_alien_species_chile[df_alien_species_chile['FirstRecord'] >= 1970]['FirstRecord'].max()}",
    nbins=50,
)
fig.show()

# Alien Species

In [10]:
df_alien_species_chile["FirstRecordDate"] = (
    df_alien_species_chile["FirstRecord"]
    .apply(lambda x: str(x) + "-01-01")
    .apply(np.datetime64)
)

# Data Chile

In [ ]:
def get_data(dir: Path | str) -> pd.DataFrame:
    if isinstance(dir, str):
        dir = Path(dir)

    df_distribution = pd.read_csv(dir / "distribution.txt", sep="\t")
    df_distribution.set_index("id", inplace=True)

    df_taxon = pd.read_csv(dir / "taxon.txt", sep="\t")
    df_taxon.set_index("id", inplace=True)

    df_species_profile = pd.read_csv(dir / "speciesprofile.txt", sep="\t")
    df_species_profile.set_index("id", inplace=True)

    return pd.concat([df_taxon, df_distribution, df_species_profile], axis=1)


df_chile_griis = get_data(DATA_DIR / "chile-griis-gbif")
df_chile_griis["source"] = "chile-griis-gbif"

df_chile_juan_fernandez = get_data(DATA_DIR / "juan-fernandez-islands-chile-griis")
df_chile_juan_fernandez["source"] = "juan-fernandez-islands-chile-griis"

df_chile_rapa_nui = get_data(DATA_DIR / "rapa-nui-chile-griis-gbif")
df_chile_rapa_nui["source"] = "rapa-nui-chile-griis-gbif"

# Cálculo tasa especies invasoras

In [22]:
df_alien_species_chile["TaxonNameId"] = df_alien_species_chile["TaxonName"].str.lower()

In [28]:
df_griis = pd.concat([df_chile_griis, df_chile_juan_fernandez, df_chile_rapa_nui])
df_griis["id_name"]: str = None
for i, r in df_griis.iterrows():
    if isinstance(r["acceptedNameUsage"], str):
        # print(f'{" ".join(r["acceptedNameUsage"].split(" ")[:2])}')
        df_griis.at[i, "id_name"] = (
            f'{" ".join(r["acceptedNameUsage"].split(" ")[:2]).lower()}'
        )
    else:
        # print(f'{" ".join(r["scientificName"].split(" ")[:2])}')
        df_griis.at[i, "id_name"] = (
            f'{" ".join(r["scientificName"].split(" ")[:2]).lower()}'
        )

In [29]:
df_6_1_especies_invasoras = pd.merge(
    left=df_alien_species_chile[
        ["TaxonNameId", "TaxonName", "FirstRecordDate", "FirstRecord"]
    ],
    right=df_griis[["id_name", "isInvasive"]],
    how="outer",
    left_on="TaxonNameId",
    right_on="id_name",
).reset_index(drop=True)

df_6_1_especies_invasoras.drop_duplicates(inplace=True)

In [30]:
df_6_1_especies_invasoras.reset_index(drop=True, inplace=True)

In [31]:
griis = 0
joined = 0
zenodo = 0
for i, r in df_6_1_especies_invasoras.iterrows():
    if r["TaxonNameId"] == r["id_name"]:
        joined += 1
    elif isinstance(r["TaxonNameId"], float) and r["id_name"]:
        griis += 1
    elif r["TaxonNameId"] and isinstance(r["id_name"], float):
        zenodo += 1

print(f"Solo GRIIS: {griis} | Joined: {joined} | Solo introducidas: {zenodo}")

Solo GRIIS: 301 | Joined: 684 | Solo introducidas: 365


In [34]:
df_6_1_especies_invasoras["isInvasiveNum"] = df_6_1_especies_invasoras[
    "isInvasive"
].apply(lambda x: 1 if x == "invasive" else 0)

In [35]:
N_INVASIVE_SPECIES = df_6_1_especies_invasoras.isInvasiveNum.sum()

In [38]:
df_6_1_especies_invasoras_v2 = (
    df_6_1_especies_invasoras[df_6_1_especies_invasoras["FirstRecord"] >= 1970]
    .groupby("FirstRecord")
    .agg({"TaxonName": "count", "isInvasiveNum": "sum"})
    .reset_index()
)
df_6_1_especies_invasoras_v2["rate"] = (
    df_6_1_especies_invasoras_v2["isInvasiveNum"]
    / df_6_1_especies_invasoras_v2["TaxonName"]
    * 100
)

df_6_1_especies_invasoras_v2["rate2"] = (
    df_6_1_especies_invasoras_v2["isInvasiveNum"] / N_INVASIVE_SPECIES * 100
)
df_6_1_especies_invasoras_v2["rate2"] = df_6_1_especies_invasoras_v2["rate2"].round(2)

In [40]:
y_data = (
    df_6_1_especies_invasoras_v2[["TaxonName", "isInvasiveNum"]]
    .groupby(df_6_1_especies_invasoras_v2.index // 5)
    .sum()
)
y_data["rate"] = y_data["isInvasiveNum"] / y_data["TaxonName"] * 100
y_data["rate"] = y_data["rate"].round(2)
y_data["rate2"] = y_data["isInvasiveNum"] / N_INVASIVE_SPECIES * 100
y_data["rate2"] = y_data["rate2"].round(2)

x_labels = (
    df_6_1_especies_invasoras_v2["FirstRecord"]
    .groupby(df_6_1_especies_invasoras_v2.index // 5)
    .first()
)
df_6_1_especies_invasoras_v3 = y_data.join(x_labels)
# df_6_1_especies_invasoras_v2.rolling(5).sum()

In [41]:
fig = px.line(
    df_6_1_especies_invasoras_v2, x="FirstRecord", y="rate2", title="Tasa de invasión"
)
fig.show()

In [42]:
fig = px.line(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate2",
    title=f"Tasa de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v3['FirstRecord'].max():.0f}",
)
fig.show()

In [43]:
fig = px.bar(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate2",
    title=f"Tasa de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v2['FirstRecord'].max():.0f}",
    text_auto=True,
)
fig.show()

In [44]:
fig = px.bar(
    df_6_1_especies_invasoras_v3,
    x="FirstRecord",
    y="rate",
    title=f"Tasa de especies invasoras entre {df_6_1_especies_invasoras_v3['FirstRecord'].min():.0f} y {df_6_1_especies_invasoras_v2['FirstRecord'].max():.0f}",
    text_auto=True,
)
fig.show()